In [2]:
import pandas as pd

In [4]:
csv_path = "../data/hotels.csv"
df = pd.read_csv(csv_path)

In [6]:
df.shape

(119390, 32)

In [8]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [11]:
df.dtypes.to_frame("dtype")


,dtype
hotel,str
is_canceled,int64
lead_time,int64
arrival_date_year,int64
arrival_date_month,str
arrival_date_week_number,int64
arrival_date_day_of_month,int64
stays_in_weekend_nights,int64
stays_in_week_nights,int64
adults,int64


In [14]:
df["reservation_status_date"] = pd.to_datetime(df["reservation_status_date"])


In [15]:
df["children"] = df["children"].astype("Int64")

In [16]:
for col in ["agent", "company"]:
    df[col] = df[col].astype("Int64").astype("string")

In [17]:
df.dtypes.to_frame("dtype")


,dtype
hotel,str
is_canceled,int64
lead_time,int64
arrival_date_year,int64
arrival_date_month,str
arrival_date_week_number,int64
arrival_date_day_of_month,int64
stays_in_weekend_nights,int64
stays_in_week_nights,int64
adults,int64


In [19]:
missing = df.isna().sum()
missing[missing > 0]

children         4
country        488
agent        16340
company     112593
dtype: int64

In [22]:
print(df["children"].value_counts())

children
0     110800
1       4861
2       3652
3         76
10         1
Name: count, dtype: Int64


In [ ]:
for col in ["country", "agent", "company"]:
    print(col, "->", df[col].nunique(), "distinct values")
    print(df[col].value_counts().head())
    print()

country -> 178 distinct values
country
PRT    48590
GBR    12129
FRA    10415
ESP     8568
DEU     7287
Name: count, dtype: int64

agent -> 334 distinct values
agent
9           31961
No agent    16340
240         13922
1            7191
14           3640
Name: count, dtype: Int64

company -> 353 distinct values
company
No company    112593
40               927
223              784
67               267
45               250
Name: count, dtype: Int64

children -> 5 distinct values
children
0     110800
1       4861
2       3652
3         76
10         1
Name: count, dtype: Int64



In [20]:
df["children"] = df["children"].fillna(0)
df["country"] = df["country"].fillna("Unknown")
df["agent"] = df["agent"].fillna("No agent")
df["company"] = df["company"].fillna("No company")

In [23]:
n_duplicates = df.duplicated().sum()
print(n_duplicates)

31994


In [24]:
df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(6)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
40772,City Hotel,0,0,2015,August,32,7,0,2,2,...,No Deposit,14,No company,0,Transient,75.0,0,1,Check-Out,2015-08-09
40802,City Hotel,0,0,2015,August,32,7,0,2,2,...,No Deposit,14,No company,0,Transient,75.0,0,1,Check-Out,2015-08-09
40821,City Hotel,0,0,2015,August,32,8,0,1,2,...,No Deposit,9,No company,0,Transient,89.0,0,1,Check-Out,2015-08-09
40838,City Hotel,0,0,2015,August,32,8,0,1,2,...,No Deposit,9,No company,0,Transient,89.0,0,1,Check-Out,2015-08-09
76792,City Hotel,0,0,2015,August,33,10,1,0,2,...,No Deposit,No agent,No company,0,Transient,75.0,0,0,Check-Out,2015-08-11
76793,City Hotel,0,0,2015,August,33,10,1,0,2,...,No Deposit,No agent,No company,0,Transient,75.0,0,0,Check-Out,2015-08-11


In [25]:
df = df.drop_duplicates().reset_index(drop=True)

### Duplicates: conclusion and action

**Conclusion**
- 31,994 rows (26.8 %) are exact duplicates across all 32 columns.
- There is no booking ID, so we cannot prove whether they are repeated records or real bookings.
- Duplicates are concentrated in the `Groups` segment (75 % of its rows) and are cancelled more often (63 % vs 27.5 %). Some may be real multi-room group bookings.
- Keeping them could repeat the same row in the training and test sets and inflate model performance.

**Action**
- Remove the duplicates and keep the first copy of each row.
- The dataset goes from 119,390 to 87,396 rows.
- The cancellation rate changes from 37.0 % to 27.5 %.

In [26]:
total_nights = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
total_guests = df["adults"] + df["children"] + df["babies"]


print("adr < 0:", (df["adr"] < 0).sum())
print("adr == 0:", (df["adr"] == 0).sum())
print("zero nights:", (total_nights == 0).sum())
print("zero guests:", (total_guests == 0).sum())


print(df[["adults", "children", "babies"]].max())

adr < 0: 1
adr == 0: 1778
zero nights: 651
zero guests: 166
adults      55
children    10
babies      10
dtype: Int64


In [27]:
df[df["adr"] < 0].T

,11591
hotel,Resort Hotel
is_canceled,0
lead_time,195
arrival_date_year,2017
arrival_date_month,March
arrival_date_week_number,10
arrival_date_day_of_month,5
stays_in_weekend_nights,4
stays_in_week_nights,6
adults,2


In [28]:
df = df[df["adr"] >= 0].reset_index(drop=True)
print(df.shape)

(87395, 32)


In [30]:
total_nights = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]

zero_nights = df[total_nights == 0]
print(len(zero_nights))


print(zero_nights["reservation_status"].value_counts())
print(zero_nights["market_segment"].value_counts())
print(zero_nights["adr"].describe())

651
reservation_status
Check-Out    622
Canceled      16
No-Show       13
Name: count, dtype: int64
market_segment
Online TA        297
Direct           131
Offline TA/TO    122
Corporate         47
Groups            34
Complementary     15
Aviation           5
Name: count, dtype: int64
count    651.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: adr, dtype: float64


In [31]:
df = df[total_nights != 0].reset_index(drop=True)
print(df.shape)

(86744, 32)


In [32]:
total_guests = df["adults"] + df["children"] + df["babies"]


zero_guests = df[total_guests == 0]
print(len(zero_guests))

print(zero_guests["reservation_status"].value_counts())
print(zero_guests["market_segment"].value_counts())
print((zero_guests["adr"] == 0).sum())

106
reservation_status
Check-Out    96
Canceled      9
No-Show       1
Name: count, dtype: int64
market_segment
Online TA        36
Direct           21
Offline TA/TO    18
Groups           16
Corporate        10
Complementary     5
Name: count, dtype: int64
75


In [33]:
df = df[total_guests != 0].reset_index(drop=True)
print(df.shape)

(86638, 32)


In [34]:

extreme = df[(df["adults"] >= 10) | (df["children"] >= 9) | (df["babies"] >= 9)]
print(len(extreme))

extreme[["hotel", "adults", "children", "babies", "market_segment", "adr", "is_canceled"]]

16


,hotel,adults,children,babies,market_segment,adr,is_canceled
314,Resort Hotel,2,10,0,Offline TA/TO,133.16,1
1426,Resort Hotel,40,0,0,Direct,0.00,1
1474,Resort Hotel,26,0,0,Offline TA/TO,0.00,1
1528,Resort Hotel,50,0,0,Direct,0.00,1
1634,Resort Hotel,26,0,0,Offline TA/TO,0.00,1
1757,Resort Hotel,26,0,0,Offline TA/TO,0.00,1
1785,Resort Hotel,27,0,0,Direct,0.00,1
1828,Resort Hotel,27,0,0,Direct,0.00,1
1866,Resort Hotel,26,0,0,Offline TA/TO,0.00,1
2013,Resort Hotel,26,0,0,Offline TA/TO,0.00,1


In [35]:
# Remove bookings with extreme guest counts
df = df.drop(extreme.index).reset_index(drop=True)
print(df.shape)

(86622, 32)


In [36]:

zero_adults = df[df["adults"] == 0]
print(len(zero_adults))


print(zero_adults["hotel"].value_counts())
print(zero_adults["children"].value_counts())
print(zero_adults["reserved_room_type"].value_counts().head())
print(zero_adults["adr"].median())

219
hotel
City Hotel    219
Name: count, dtype: int64
children
2    204
3     11
1      4
Name: count, dtype: Int64
reserved_room_type
B    205
A      9
F      2
G      1
E      1
Name: count, dtype: int64
87.08


In [37]:
pd.crosstab(df["reservation_status"], df["is_canceled"])

is_canceled,0,1
reservation_status,,
Canceled,0,22973
Check-Out,62650,0
No-Show,0,999


In [38]:
# Final checks before saving
print(df.shape)
print(df.isna().sum().sum())
print(df.duplicated().sum())
print((df["adr"] < 0).sum())

(86622, 32)
0
0
0


In [39]:
# Save the cleaned dataset
clean_path = "../data/hotels_clean.csv"
df.to_csv(clean_path, index=False)

In [40]:
# Reload the saved file and check it
check = pd.read_csv(clean_path)
print(check.shape)

(86622, 32)
